# Rectified Flow 动漫头像：Google Colab 训练

这个 Notebook 用于在 Google Colab 的 Tesla T4 上运行本项目。代码和数据会从 Google Drive 的 ZIP 解压到 `/content`，避免逐张从 Drive 读取图片；checkpoint、生成图片和 TensorBoard 日志会直接保存在 Drive，断线后仍可恢复。

使用前请在 Colab 菜单中选择：`代码执行程序 → 更改运行时类型 → T4 GPU`，然后从上到下运行单元格。第一次建议把目标 epoch 设置为 1，完成冒烟验证后再改成 100。

## 1. 挂载 Google Drive

In [3]:
from google.colab import drive
drive.mount('/content/drive')

## 2. 配置 ZIP 路径和训练参数

只需要修改 `DRIVE_ZIP_PATH`，使它与云端硬盘里的压缩包路径一致。`DRIVE_SAVE_DIR` 用于持久化训练结果。再次连接 Colab 时保持相同的保存目录，即可自动恢复。

In [4]:
from pathlib import Path

# 示例：压缩包直接放在“我的云端硬盘”根目录。请按实际文件名修改。
DRIVE_ZIP_PATH = Path('//content/drive/MyDrive/KRM_RF_Anime Images.zip')

# checkpoint、样本、loss 图和 TensorBoard 日志的持久化目录。
DRIVE_SAVE_DIR = Path('/content/drive/MyDrive/KRM_RF_Anime_Colab_Results')

# 这是训练的总目标 epoch，而不是本次额外训练多少个 epoch。
# 首次建议设为 1；验证成功后改为 100。已有 latest.pt 时会自动续训。
TARGET_EPOCHS = 100
BATCH_SIZE = 16
NUM_WORKERS = 2

assert DRIVE_ZIP_PATH.is_file(), f'找不到压缩包，请修改 DRIVE_ZIP_PATH：{DRIVE_ZIP_PATH}'
DRIVE_SAVE_DIR.mkdir(parents=True, exist_ok=True)
print('压缩包：', DRIVE_ZIP_PATH)
print('持久化目录：', DRIVE_SAVE_DIR)
print('训练目标 epoch：', TARGET_EPOCHS)

## 3. 解压项目到 Colab 临时磁盘

每次新的 Colab 会话都需要重新解压。该操作不会修改 Drive 中的 ZIP。Notebook 会自动寻找包含 `train.py` 和 `config/default.yaml` 的项目目录。

In [5]:
import shutil
import zipfile

EXTRACT_ROOT = Path('/content/krm_rf_workspace')
if EXTRACT_ROOT.exists():
    shutil.rmtree(EXTRACT_ROOT)
EXTRACT_ROOT.mkdir(parents=True)

with zipfile.ZipFile(DRIVE_ZIP_PATH) as archive:
    root = EXTRACT_ROOT.resolve()
    for member in archive.infolist():
        destination = (EXTRACT_ROOT / member.filename).resolve()
        if destination != root and root not in destination.parents:
            raise RuntimeError(f'压缩包包含不安全路径：{member.filename}')
    archive.extractall(EXTRACT_ROOT)

project_candidates = [
    path.parent for path in EXTRACT_ROOT.rglob('train.py')
    if (path.parent / 'config' / 'default.yaml').is_file()
]
if len(project_candidates) != 1:
    raise RuntimeError(
        f'应找到一个项目目录，实际找到 {len(project_candidates)} 个：{project_candidates}'
    )
PROJECT_DIR = project_candidates[0]
DATA_DIR = PROJECT_DIR / 'Data' / 'train' / 'nolabel'
assert DATA_DIR.is_dir(), f'压缩包中缺少数据目录：{DATA_DIR}'

# 兼容旧版压缩包：Windows 曾把 Python 的 data 包与 Data 数据目录合并。
# Linux 区分大小写，因此为旧代码建立临时小写链接；新版项目使用 datasets 包。
legacy_data_package = PROJECT_DIR / 'Data'
compat_data_package = PROJECT_DIR / 'data'
if (legacy_data_package / '__init__.py').is_file() and not compat_data_package.exists():
    compat_data_package.symlink_to(legacy_data_package, target_is_directory=True)
    print('已启用旧版 data/Data 大小写兼容链接')

image_count = sum(1 for _ in DATA_DIR.glob('*.png'))
assert image_count > 0, f'数据目录没有 PNG：{DATA_DIR}'

%cd {PROJECT_DIR}
print('项目目录：', PROJECT_DIR)
print('PNG 数量：', image_count)

## 4. 检查 GPU 和 PyTorch

必须看到 `CUDA 可用：True`。如果是 False，请在 Colab 中切换到 T4 GPU 后重新连接。

In [6]:
import subprocess
import torch

print('PyTorch：', torch.__version__)
print('CUDA 可用：', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('当前没有 GPU。请选择 T4 GPU 运行时并重新连接。')
print('GPU：', torch.cuda.get_device_name(0))
print('显存（GB）：', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
subprocess.run(['nvidia-smi'], check=True)

## 5. 安装 Colab 需要的依赖

这里保留 Colab 自带的 `torch` 和 `torchvision`，避免下载大型 CUDA 包或造成版本冲突。

In [7]:
%pip install -q PyYAML==6.0.2 matplotlib==3.10.5 tensorboard==2.20.0 moviepy==1.0.3 tqdm==4.67.1 pytest==8.4.1

import torchvision
import yaml
print('torch：', torch.__version__)
print('torchvision：', torchvision.__version__)
print('依赖安装完成')

## 6. 生成 Colab 专用配置

训练图片保存在 `/content`，提高小图片读取速度；划分清单、权重、样本、loss 图和 TensorBoard 日志保存在 Drive。`resume: true` 会自动读取 Drive 中的 `latest.pt`。

In [8]:
import yaml

with (PROJECT_DIR / 'config' / 'default.yaml').open('r', encoding='utf-8') as file:
    config = yaml.safe_load(file)

config['data']['raw_dir'] = str(DATA_DIR)
config['data']['split_dir'] = str(DRIVE_SAVE_DIR / 'data_splits')
config['data']['num_workers'] = NUM_WORKERS
config['training']['device'] = 'cuda'
config['training']['epochs'] = TARGET_EPOCHS
config['training']['batch_size'] = BATCH_SIZE
config['training']['mixed_precision'] = True
config['training']['resume'] = True
config['paths']['output_dir'] = str(DRIVE_SAVE_DIR / 'outputs')
config['paths']['checkpoint_dir'] = str(DRIVE_SAVE_DIR / 'outputs' / 'checkpoints')
config['paths']['sample_dir'] = str(DRIVE_SAVE_DIR / 'outputs' / 'samples')
config['paths']['plot_dir'] = str(DRIVE_SAVE_DIR / 'outputs' / 'plots')
config['paths']['log_dir'] = str(DRIVE_SAVE_DIR / 'runs' / 'rectified_flow')

COLAB_CONFIG = PROJECT_DIR / 'config' / 'colab.yaml'
with COLAB_CONFIG.open('w', encoding='utf-8') as file:
    yaml.safe_dump(config, file, allow_unicode=True, sort_keys=False)

print('Colab 配置：', COLAB_CONFIG)
print('batch size：', config['training']['batch_size'])
print('总目标 epoch：', config['training']['epochs'])
print('checkpoint：', config['paths']['checkpoint_dir'])

## 7. 可选：运行快速单元测试

建议首次运行时执行。它不会开始训练，也不会读取全部数据。

In [9]:
subprocess.run(['python', '-m', 'pytest', '-q'], cwd=PROJECT_DIR, check=True)

## 8. 可选：启动 TensorBoard

可以在训练前启动，训练期间重新打开这个单元格的输出即可查看 loss、最终生成图、噪声到图片的轨迹网格和视频。

In [10]:
TENSORBOARD_LOG_DIR = str(DRIVE_SAVE_DIR / 'runs')
%load_ext tensorboard
%tensorboard --logdir $TENSORBOARD_LOG_DIR --port 6006

## 9. 开始或继续训练

这个单元格会正式训练。首次会校验并按内容哈希去重，然后生成 90%/5%/5% 清单。每个 epoch 都将 `latest.pt` 保存到 Drive。

如果 Colab 意外断线：重新打开 Notebook，从第 1 节运行到这里。只要 `DRIVE_SAVE_DIR` 相同并且目标 epoch 大于 checkpoint 中的 epoch，就会从下一个 epoch 自动继续。

In [ ]:
import subprocess
import time

training_started = time.time()

process = subprocess.Popen(
    [
        "python",
        "-u",  # 禁止 Python 缓冲输出
        "train.py",
        "--config",
        str(COLAB_CONFIG),
    ],
    cwd=PROJECT_DIR,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

assert process.stdout is not None

for line in process.stdout:
    print(line, end="", flush=True)

return_code = process.wait()

elapsed_hours = (time.time() - training_started) / 3600
print(f"\n进程退出码：{return_code}")
print(f"本次运行时间：{elapsed_hours:.2f} 小时")

if return_code != 0:
    raise RuntimeError(f"训练异常退出，退出码：{return_code}")

## 10. 查看保存结果和当前 checkpoint

这个单元格不会训练，只读取 Drive 中已经保存的结果。

In [ ]:
from IPython.display import display
from PIL import Image

checkpoint_dir = DRIVE_SAVE_DIR / 'outputs' / 'checkpoints'
latest_checkpoint = checkpoint_dir / 'latest.pt'
if latest_checkpoint.is_file():
    checkpoint_info = torch.load(latest_checkpoint, map_location='cpu', weights_only=True)
    print('已完成 epoch：', checkpoint_info['epoch'])
    print('global step：', checkpoint_info.get('global_step'))
    print('最佳验证 loss：', checkpoint_info.get('best_val_loss'))
else:
    print('目前没有 latest.pt')

sample_files = sorted((DRIVE_SAVE_DIR / 'outputs' / 'samples').glob('*.png'))
if sample_files:
    print('最新样本：', sample_files[-1])
    display(Image.open(sample_files[-1]))
else:
    print('还没有生成预览；默认每 5 个 epoch 生成一次。')

## 11. 使用最佳模型独立生成图片

至少完成一个验证 epoch 后会有 `best.pt`。修改种子可以生成不同图片。

In [ ]:
SAMPLE_SEED = 123
BEST_CHECKPOINT = DRIVE_SAVE_DIR / 'outputs' / 'checkpoints' / 'best.pt'
assert BEST_CHECKPOINT.is_file(), f'找不到最佳模型：{BEST_CHECKPOINT}'

subprocess.run(
    [
        'python', 'sample.py', '--config', str(COLAB_CONFIG),
        '--checkpoint', str(BEST_CHECKPOINT), '--seed', str(SAMPLE_SEED),
    ],
    cwd=PROJECT_DIR,
    check=True,
)
generated_path = DRIVE_SAVE_DIR / 'outputs' / 'samples' / f'sample_seed_{SAMPLE_SEED}.png'
display(Image.open(generated_path))

## 12. 可选：测试集评估

训练完成或训练到一个阶段后，可以计算测试集 flow-matching MSE。

In [ ]:
subprocess.run(
    ['python', 'evaluate.py', '--config', str(COLAB_CONFIG), '--checkpoint', str(BEST_CHECKPOINT)],
    cwd=PROJECT_DIR,
    check=True,
)

## 免费运行时长建议

- 首次先用 `TARGET_EPOCHS = 1` 验证完整流程，并记录一个 epoch 的时间。
- 验证正常后，将 `TARGET_EPOCHS` 改成 30、60 或 100。该值是最终目标，不是追加数量。
- `latest.pt` 每个 epoch 都保存到 Drive，因此正常情况下只会损失断线时正在训练的那个 epoch。
- 训练期间不要关闭浏览器标签页，建议保持电脑联网。
- 若显存不足，将 `BATCH_SIZE` 改成 8。
- Colab 临时磁盘会在会话结束后清空，这是正常的；持久化结果位于 `DRIVE_SAVE_DIR`。